<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-7-to-10/blob/main/09_Modeling_Probabilities_and_Nonlinearities.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **What is an Activation Function?**

*Activation function* (fungsi aktivasi) adalah fungsi matematika yang diterapkan pada nodus/neuron di dalam layer jaringan saraf selama proses *forward propagation* (prediksi). Fungsi ini menerima satu angka masukan dan mengembalikannya ke dalam bentuk angka baru. Namun, dari tak hingga banyaknya fungsi di alam semesta, hanya fungsi yang memenuhi kriteria tertentu yang dapat digunakan sebagai fungsi aktivasi. Kriteria utama fungsi aktivasi yang baik:
1. **Kontinu dan Memiliki Domain Tak Hingga** (*continuous and infinite in domain*).
2. **Monoton** (*monotonic* / tidak bolak-balik naik turun).
3. **Non-linier** (*nonlinear* / memiliki lengkungan atau belokan).
4. **Komputasi Efisien** (*efficiently computable* beserta turunannya).

# **Standard Hidden-Layer Activation Functions (Sigmoid &amp; Tanh)**

* Sigmoid ($y = \frac{1}{1 + e^{-x}}$): Mengompresi (squishes) rentang angka masukan dari $-\infty$ hingga $+\infty$ secara mulus ke dalam rentang interval $(0, 1)$. Keunggulannya adalah nilainya dapat diinterpretasikan sebagai probabilitas.
* Tanh ($y = \tanh(x)$): Mirip dengan Sigmoid, tetapi mengompresi angka ke dalam rentang $(-1, 1)$. Tanh jauh lebih unggul dibanding Sigmoid pada hidden layer karena mampu memberikan korelasi positif maupun korelasi negatif (negative correlation).

In [ ]:
import numpy as np

# ReLU & Derivatif
def relu(x):
    return (x >= 0) * x

def relu2deriv(output):
    return output >= 0

# Sigmoid & Derivatif
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid2deriv(output):
    return output * (1 - output)

# Tanh & Derivatif
def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

# Softmax (Multiclass Output)
def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

# **Standard Output-Layer Activation Functions**

Pemilihan fungsi aktivasi pada *output layer* sangat bergantung pada tipe target yang ingin diprediksi:
1. **Tanpa Fungsi Aktivasi**: Digunakan untuk memprediksi angka kontinu murni (seperti prediksi suhu udara) tanpa membatasi keluaran antara 0 dan 1.
2. **Sigmoid**: Digunakan untuk memprediksi **probabilitas biner independen** (*unrelated yes/no probabilities*) di mana satu nodus tidak memengaruhi nodus keluaran lainnya.
3. **Softmax**: Digunakan untuk memprediksi **probabilitas eksklusif multikelas** (*which-one probabilities*).

# **The Core Issue: Inputs Have Similarity**

Gambar digit MNIST memiliki piksel-piksel yang saling tumpang-tindih (misalnya angka 2 dan 3 memiliki goresan piksel yang serupa). Pada jaringan linier, masukan yang mirip akan menghasilkan prediksi keluaran yang mirip pula.

# **Softmax Computation**

Softmax mengatasi masalah kemiripan masukan dengan cara menonjolkan nilai terbesar dan menekan nilai kecil. Tahapannya:
* Meng-eksponensialkan setiap masukan ($e^x$), yang mengubah angka 0 menjadi 1 dan angka besar menjadi nilai yang sangat raksasa.
* Membagi setiap nilai dengan total penjumlahan seluruh elemen ($e^x / \sum e^x$). Hasil akhirnya adalah distribusi probabilitas di mana total seluruh nilai nodus keluaran sama dengan $1.0$ ($100\%$).

# **Activation Installation Instructions**

Memasang fungsi aktivasi pada *forward propagation* sangat mudah, yaitu mengalikan masukan mentah layer (`np.dot(layer_0, weights_0_1)`) dengan fungsi aktivasi. Namun, pada *backpropagation*, sinyal eror (`delta`) harus disesuaikan dengan dikalikan turunan/kemiringan (*slope*) dari fungsi aktivasi pada titik prediksi tersebut.

# **Multiplying Delta by the Slope &amp; Converting Output to Slope (Derivative)**

Kemiringan (*slope*) mengukur seberapa sensitif perubahan masukan memengaruhi keluaran nodus. Jika pergeseran bobot tidak memberi dampak pada nodus (misalnya pada area datar Sigmoid/ReLU), nilai kemiringan mendekati 0 sehingga `delta` dikalikan 0 agar bobot tidak diubah.

**Trik Turunan Praktis**: Kebanyakan fungsi aktivasi memungkinkan perhitungan turunan secara langsung dari nilai output-nya pada forward propagation:
* $\text{Sigmoid}' = \text{output} \times (1 - \text{output})$
* $\text{Tanh}' = 1 - \text{output}^2$

# **Upgrading the MNIST Network**

 Meng-upgrade jaringan MNIST dengan mengganti aktivasi hidden layer menjadi Tanh dan output layer menjadi Softmax. Karena Tanh peka terhadap skala masukan, deviasi standar inisialisasi bobot awal diperkecil dari range $[-0.1, 0.1]$ menjadi $[-0.01, 0.01]$. Pembaruan ini berhasil meningkatkan akurasi pengujian MNIST dari $81.8\%$ menjadi $87.0\%$.

In [ ]:
import sys, numpy as np

np.random.seed(1)

# Menggunakan data MNIST (images & labels)
alpha, iterations, hidden_size = (2, 300, 100)
pixels_per_image, num_labels = (784, 10)
batch_size = 100

# Penyesuaian skala inisialisasi bobot khusus Tanh (-0.01 hingga 0.01)
weights_0_1 = 0.02 * np.random.random((pixels_per_image, hidden_size)) - 0.01
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))

        # 1. Forward Propagation dengan Tanh & Softmax
        layer_0 = images[batch_start:batch_end]
        layer_1 = tanh(np.dot(layer_0, weights_0_1))

        dropout_mask = np.random.randint(2, size=layer_1.shape)
        layer_1 *= dropout_mask * 2

        layer_2 = softmax(np.dot(layer_1, weights_1_2))

        for k in range(batch_size):
            correct_cnt += int(np.argmax(layer_2[k:k+1]) == \\
                               np.argmax(labels[batch_start+k:batch_start+k+1]))

        # 2. Backpropagation & Slope Multiplication
        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)
        layer_1_delta *= dropout_mask

        # 3. Update Bobot
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)
        weights_0_1 += alpha * layer_0.T.dot(layer_1_delta)

    test_correct_cnt = 0
    for i in range(len(test_images)):
        layer_0 = test_images[i:i+1]
        layer_1 = tanh(np.dot(layer_0, weights_0_1))
        layer_2 = np.dot(layer_1, weights_1_2)
        test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))

    if (j % 10 == 0):
        sys.stdout.write("\\n" + \\
            "I:" + str(j) + \\
            " Test-Acc:" + str(test_correct_cnt / float(len(test_images))) + \\
            " Train-Acc:" + str(correct_cnt / float(len(images))))